# Vetorização dos trechos no Google Colab (EmbeddingGemma 2)

Use este notebook quando a sua máquina não tem GPU. Ele lê a entrada gerada por `preparar_embed.py`, gera os
vetores numa GPU do Colab e grava o resultado no seu Google Drive.

**Antes de rodar**
1. No seu PC: `python preparar_embed.py dados/embed_in` → gera `dados/embed_in.jsonl.gz` e `dados/embed_in.ids.npy`.
2. Suba **só** o `embed_in.jsonl.gz` para a pasta `MyDrive/busca-embed/` do seu Google Drive.
3. Aqui: *Ambiente de execução → Alterar tipo* → GPU (**L4** ou **A100**; a T4 funciona, porém mais devagar).
4. *Ambiente de execução → Executar tudo*.

**Depois**: baixe `MyDrive/busca-embed/embed_in.npy` para a pasta `dados/` do projeto e rode
`python construir_indice.py vetores dados/embed_in.npy dados/embed_in.ids.npy`.

> Privacidade: o texto dos seus documentos passa pelo seu Drive e pela sua VM do Colab. Se isso não for aceitável
> para algum material, vetorize localmente (`embed_local.py`).

**Retomável**: cada fatia de 10 mil trechos é salva no Drive assim que fica pronta. Se a sessão cair, rode de novo
— as fatias prontas são puladas. Ao terminar, **desconecte a sessão** (o Colab cobra enquanto a GPU estiver ligada).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

PASTA = "/content/drive/MyDrive/busca-embed"   # onde você subiu o arquivo
NOME = "embed_in"                              # embed_in.jsonl.gz → embed_in.npy
FATIA = 10_000

In [ ]:
!pip install -q -U "sentence-transformers>=6.1.0" "transformers>=5.19.0"

In [ ]:
import gzip, json, os, time
import numpy as np, torch
from sentence_transformers import SentenceTransformer

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# NUNCA float16: o EmbeddingGemma 2 estoura a faixa numérica e gera NaN. Use bf16 (L4/A100) ou fp32 (T4).
dt = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
m = SentenceTransformer("google/embeddinggemma-2", device="cuda", model_kwargs={"torch_dtype": dt},
                        config_kwargs={"vision_config": None, "audio_config": None})   # só o módulo de texto (270M)
m.max_seq_length = 1024
print(torch.cuda.get_device_name(0), dt)

In [ ]:
textos = []
with gzip.open(f"{PASTA}/{NOME}.jsonl.gz", "rt", encoding="utf-8") as f:
    for l in f:
        r = json.loads(l)
        textos.append("title: " + (r.get("t") or "none") + " | text: " + r["text"])   # prompt de documento
n = len(textos); nf = (n + FATIA - 1) // FATIA
partes = f"{PASTA}/partes_{NOME}"; os.makedirs(partes, exist_ok=True)
lote = 128 if "A100" in torch.cuda.get_device_name(0) else 64   # 128 estoura a L4 (22 GB)
print(f"{n} trechos em {nf} fatias")
for k in range(nf):
    arq = f"{partes}/parte_{k:04d}.npy"
    if os.path.exists(arq):
        continue
    t0 = time.time()
    e = m.encode(textos[k*FATIA:(k+1)*FATIA], batch_size=lote, normalize_embeddings=True, convert_to_numpy=True)
    e = np.asarray(e, dtype=np.float32)
    assert np.isfinite(e).all(), "NaN nos vetores"
    np.save(arq, e.astype(np.float16))
    print(f"[{k+1}/{nf}] {min((k+1)*FATIA, n)}/{n} em {time.time()-t0:.0f}s")

In [ ]:
emb = np.concatenate([np.load(f"{partes}/parte_{k:04d}.npy") for k in range(nf)])
assert emb.shape[0] == n
np.save(f"{PASTA}/{NOME}.npy", emb)
print("OK:", emb.shape, "→", f"{PASTA}/{NOME}.npy  (baixe para a pasta dados/ do projeto)")